<a href="https://colab.research.google.com/github/SiramGanesh/ML-DEEP-LEARNING-QUANTUM-ML/blob/main/randomforest_and_deeplearning_cic_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import os
from google.colab import drive

# Mount Google Drive, force remount to handle potential connection issues
drive.mount('/content/drive', force_remount=True)

# Define the path to your folder
folder_path = '/content/drive/My Drive/archive'

# Check if the folder exists
if not os.path.exists(folder_path):
    print(f"Error: Folder '{folder_path}' not found. Please check the folder name and path in your Google Drive.")
else:
    print(f"Contents of '{folder_path}':")
    files = os.listdir(folder_path)
    if files:
        for f in files:
            print(f)

        # Assuming the data is in a CSV file, let's try to find and load the first one
        csv_files = [f for f in files if f.endswith('.csv')]
        if csv_files:
            file_to_load = os.path.join(folder_path, csv_files[0])
            print(f"\nAttempting to load '{file_to_load}' into DataFrame 'df'...")
            try:
                df = pd.read_csv(file_to_load)
                print("DataFrame 'df' loaded successfully. First 5 rows:")
                print(df.head())

                print("Shape:", df.shape)

                print("\nColumns:")
                print(df.columns.tolist())

                print("\nLabel Distribution:")
                print(df['label'].value_counts().head(10))

                print("\nTotal Classes in this file:")
                print(df['label'].nunique())

            except Exception as e:
                print(f"Error loading CSV file: {e}")
        else:
            print("No CSV files found in the folder. Please specify the correct file type or name.")
    else:
        print("The folder is empty.")

Mounted at /content/drive
Contents of '/content/drive/My Drive/archive':
part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00004-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00006-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00005-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00007-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00008-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00011-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00010-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00009-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00013-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00012-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00014-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00015-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
part-00017-363d

In [ ]:
dfs = []

SAMPLES_PER_FILE = 6000  # can increase later

for i, file in enumerate(files):

    # Construct the full path to the file
    full_file_path = os.path.join(folder_path, file)
    temp = pd.read_csv(full_file_path)

    sample_size = min(SAMPLES_PER_FILE, len(temp))

    temp = temp.sample(
        n=sample_size,
        random_state=42
    )

    dfs.append(temp)

    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/{len(files)} files")

df = pd.concat(dfs, ignore_index=True)

print("\nFinal Shape:", df.shape)

print("\nNumber of Classes:")
print(df['label'].nunique())

print("\nTop 10 Labels:")
print(df['label'].value_counts().head(10))

Processed 20/169 files
Processed 40/169 files


In [ ]:
print("Total Classes:", df['label'].nunique())

print("\nSmallest Classes:")
print(df['label'].value_counts().tail(15))

print("\nLargest Classes:")
print(df['label'].value_counts().head(15))

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["multi_label"] = le.fit_transform(df["label"])

print("Number of classes:", df["multi_label"].nunique())

print("\nEncoded Classes:")

for i, cls in enumerate(le.classes_):
    print(i, "->", cls)

In [ ]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nInfinite Values:")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

print("\nData Types:")
print(df.dtypes)

In [ ]:
feature_cols = [col for col in df.columns
                if col not in ["label",
                               "multi_label"]]

print("Number of Features:", len(feature_cols))

print(feature_cols)

In [ ]:
X = df[feature_cols].copy()
y_multi = df["multi_label"].copy()

print("X Shape:", X.shape)
print("Multiclass Labels:", y_multi.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_multi,
    test_size=0.30,
    random_state=42,
    stratify=y_multi
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nClasses in Train:", y_train.nunique())
print("Classes in Validation:", y_val.nunique())
print("Classes in Test:", y_test.nunique())

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()

# Record the start time for training
start_time = time.time()

# Train the random forest classifier
rf_model.fit(X_train, y_train)

# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
training_time = end_time - start_time

In [ ]:
start_time = time.time()

# Make predictions on the validation data
y_pred = rf_model.predict(X_val)

# Record the end time for testing
end_time = time.time()

# Calculate the elapsed time for testing
val_time = end_time - start_time

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='macro')
recall = recall_score(y_val, y_pred, average='macro')
f1 = f1_score(y_val, y_pred, average='macro')

print("Random Forest Accuracy:", accuracy)
# Convert seconds to minutes and seconds
mins, secs = divmod(training_time, 60)
# Print in a readable format
print(f"Training completed in {int(mins)} minutes and {secs:.2f} seconds")
# Convert seconds to minutes and seconds
mins, secs = divmod(val_time, 60)
# Print in a readable format
print(f"Training completed in {int(mins)} minutes and {secs:.2f} seconds")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

conf_matrix = confusion_matrix(y_val, y_pred)

# 1. Create a larger figure
plt.figure(figsize=(20, 18))

# 2. Generate the heatmap
# Turn off 'annot' to avoid the text clutter
# Use a different cmap like 'Blues' which often reads better for large grids
sns.heatmap(conf_matrix, annot=False, cmap='Blues',
            linewidths=0.1, linecolor='gray', square=True)

plt.xlabel('Predicted Labels', fontsize=15)
plt.ylabel('True Labels', fontsize=15)
plt.title('Confusion Matrix (34 Classes)', fontsize=20)

plt.savefig("/kaggle/working/RF_Confusion_Matrix_Clean.pdf", bbox_inches='tight')
plt.show()

In [ ]:
#use GridSearchCV to optimize the RandomForest
from sklearn.model_selection import GridSearchCV

# 1. Initialize the base model
rf = RandomForestClassifier(random_state=42)

# 2. Define a focused parameter grid
# Note: Adding too many parameters will make this take hours
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# 3. Setup GridSearchCV
# n_jobs=-1 uses all available CPU cores to speed up the process
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='f1_macro')

# 4. Fit the model
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)

# 5. Display results
print(f"Best Parameters: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier

best_rf_model = RandomForestClassifier(max_depth = None, min_samples_split = 2, n_estimators = 100)

# Record the start time for training
start_time = time.time()

# Train the random forest classifier
best_rf_model.fit(X_train, y_train)

# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
best_rf_training_time = end_time - start_time

In [ ]:
start_time = time.time()

# Make predictions on the validation data
y_pred = best_rf_model.predict(X_val)

# Record the end time for testing
end_time = time.time()

# Calculate the elapsed time for testing
best_rf_val_time = end_time - start_time

In [ ]:
# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='macro')
recall = recall_score(y_val, y_pred, average='macro')
f1 = f1_score(y_test, y_val, average='macro')

print("Random Forest Accuracy:", accuracy)
print("Time taken for training:", best_rf_training_time, "seconds")
print("Time taken for testing:", best_rf_val_time, "seconds")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Create a larger figure
plt.figure(figsize=(20, 18))

# 2. Generate the heatmap
# Turn off 'annot' to avoid the text clutter
# Use a different cmap like 'Blues' which often reads better for large grids
sns.heatmap(conf_matrix, annot=False, cmap='Blues',
            linewidths=0.1, linecolor='gray', square=True)

plt.xlabel('Predicted Labels', fontsize=15)
plt.ylabel('True Labels', fontsize=15)
plt.title('Confusion Matrix (34 Classes)', fontsize=20)

plt.savefig("/kaggle/working/Best_RF_Confusion_Matrix_Clean.pdf", bbox_inches='tight')
plt.show()

In [ ]:
def category_extraction(df):
    # extract attack category from label
    category_dict = {
        'DDoS-ACK_Fragmentation' : 'DDoS',
        'DDoS-ICMP_Flood': 'DDoS',
        'DDoS-PSHACK_Flood': 'DDoS',
        'DDoS-RSTFINFlood': 'DDoS',
        'DDoS-SlowLoris': 'DDoS',
        'DDoS-SynonymousIP_Flood': 'DDoS',
        'DDoS-UDP_Fragmentation': 'DDoS',
        'DDoS-ICMP_Fragmentation' : 'DDoS',

        'DoS-HTTP_Flood' : 'DoS',
        'DoS-SYN_Flood' : 'DoS',
        'DoS-TCP_Flood' : 'DoS',
        'DoS-UDP_Flood' : 'DoS',


        'Mirai-greeth_flood' : 'Mirai',
        'Mirai-greip_flood' : 'Mirai',
        'Mirai-udpplain' : 'Mirai',

        'BenignTraffic' : 'Benign'
    }

    # label encoding for attack categories
    df_label_cat = df.label.apply(lambda x: category_dict.get(x))
    df['label'] = df_label_cat
    return df

less_multi_df = category_extraction(df)

In [ ]:
# Calculate the counts of each class
class_counts = less_multi_df['label'].value_counts()

# Now plot the result as a pie chart
plt.figure(figsize=(6, 6))
plt.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Class Distribution')
plt.axis('equal')
plt.savefig("/kaggle/working/ClassDistAfter.pdf")
plt.show()

In [ ]:
less_multi_df = less_multi_df.dropna(how='any',axis=0)

# Handling missing values (if any)
missing_values = less_multi_df.isnull().sum()
print("Missing values:")
print(missing_values)


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

X = less_multi_df.drop(['label'], axis=1)
y = less_multi_df['label']

# 1. Ensure NO None/NaN values remain in y
# This removes the rows causing the {None} error
mask = y.notnull()
X = X[mask]
y = y[mask]

# 2. Re-calculate the sampling strategy using ONLY the valid labels
# This ensures 'None' is never included in your dictionary keys
from collections import Counter
counts = Counter(y)
n_samples = min(counts.values())

# Create the dictionary ONLY from valid, present labels
sampling_strategy = {label: n_samples for label in counts.keys()}

# 3. Re-initialize the sampler with the clean strategy
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)

# 4. Now execute the resample
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampling successful!")
print("New class distribution:", Counter(y_resampled))

In [ ]:
# Create a new DataFrame using the output data
data = pd.DataFrame(X_resampled, columns=df.columns[:-1])  # Use columns except 'Label'
data['label'] = y_resampled  # Add the 'Label' column

In [ ]:
X = data.drop(['label'], axis=1)
y = data[['label']]

In [ ]:
print(X.columns.tolist())

In [ ]:
Xchi = X[['HTTPS', 'ack_flag_number', 'Variance', 'ICMP', 'Protocol Type',
          'TCP', 'fin_flag_number', 'UDP', 'rst_flag_number', 'psh_flag_number',
          'syn_flag_number', 'rst_count', 'Magnitue', 'Header_Length', 'Min']]

Xpca = X[['Max', 'TCP', 'fin_flag_number', 'Weight', 'IPv', 'Rate',
          'syn_flag_number', 'ICMP', 'psh_flag_number', 'UDP', 'HTTP', 'DNS',
          'Drate', 'DHCP']]

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(Xchi, y, test_size=0.2, random_state=42)

# Initialize the random forest classifier
rf_model = RandomForestClassifier()

# Record the start time for training
start_time = time.time()

# Train the random forest classifier
rf_model.fit(X_train, y_train)

# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
training_time = end_time - start_time

# Record the start time for testing
start_time = time.time()

# Make predictions on the testing data
y_pred = rf_model.predict(X_test)

# Record the end time for testing
end_time = time.time()

# Calculate the elapsed time for testing
testing_time = end_time - start_time

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("Random Forest Accuracy:", accuracy)
print("Time taken for training:", training_time, "seconds")
print("Time taken for testing:", testing_time, "seconds")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Create a confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Visualize the confusion matrix with a heatmap
plt.figure(figsize=(6, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens', linewidths=0.5, linecolor='black', square=True)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.savefig("/kaggle/working/RFChideafult.pdf")
plt.show()

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(Xpca, y, test_size=0.2, random_state=42)

# Initialize the random forest classifier
rf_model = RandomForestClassifier()

# Record the start time for training
start_time = time.time()

# Train the random forest classifier
rf_model.fit(X_train, y_train)

# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
training_time = end_time - start_time

# Record the start time for testing
start_time = time.time()

# Make predictions on the testing data
y_pred = rf_model.predict(X_test)

# Record the end time for testing
end_time = time.time()

# Calculate the elapsed time for testing
testing_time = end_time - start_time

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("Random Forest Accuracy:", accuracy)
print("Time taken for training:", training_time, "seconds")
print("Time taken for testing:", testing_time, "seconds")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Create a confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Visualize the confusion matrix with a heatmap
plt.figure(figsize=(6, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens', linewidths=0.5, linecolor='black', square=True)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.savefig("/kaggle/working/RFPcadeafult.pdf")
plt.show()

In [ ]:
model = RandomForestClassifier()
model.fit(X, y)

feature_importances = pd.Series(model.feature_importances_, index=X.columns)
top_features = feature_importances.nlargest(n=15)  # Select top 10 features

top_feature_names_complate = top_features.index

print(top_feature_names_complate)
Xrf_c = X[top_feature_names_complate]

In [ ]:
Xrf_c = X[['IAT', 'urg_count', 'Tot size', 'Protocol Type', 'Min', 'rst_count',
       'Magnitue', 'Max', 'AVG', 'Variance', 'flow_duration', 'Tot sum',
       'Header_Length', 'Std', 'TCP']]

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(Xrf_c, y, test_size=0.2, random_state=42)

# Initialize the random forest classifier
rf_model = RandomForestClassifier()

# Record the start time for training
start_time = time.time()

# Train the random forest classifier
rf_model.fit(X_train, y_train)

# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
training_time = end_time - start_time

# Record the start time for testing
start_time = time.time()

# Make predictions on the testing data
y_pred = rf_model.predict(X_test)

# Record the end time for testing
end_time = time.time()

# Calculate the elapsed time for testing
testing_time = end_time - start_time

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("Random Forest Accuracy:", accuracy)
print("Time taken for training:", training_time, "seconds")
print("Time taken for testing:", testing_time, "seconds")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Create a confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Visualize the confusion matrix with a heatmap
plt.figure(figsize=(6, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens', linewidths=0.5, linecolor='black', square=True)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.savefig("/kaggle/working/RFRFdeafult.pdf")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.datasets import make_classification

X_train, X_val, y_train, y_val = train_test_split(Xrf_c, y, test_size=0.2, random_state=42)

# Initialize the random forest classifier
rf = RandomForestClassifier()

# Train the classifier
rf.fit(X_train, y_train)

# Predict probabilities
train_probs = rf.predict_proba(X_train)
val_probs = rf.predict_proba(X_val)

# Calculate log loss
train_loss = log_loss(y_train, train_probs)
val_loss = log_loss(y_val, val_probs)

# Print the losses
print(f'Training Loss: {train_loss}')
print(f'Validation Loss: {val_loss}')

# Plot the losses as a bar chart
plt.bar(['Training Loss', 'Validation Loss'], [train_loss, val_loss])
plt.ylabel('Log Loss')
plt.title('Training and Validation Loss for Random Forest with Default Parameters')
plt.savefig('training_validation_loss_comparison.png')
plt.show()

**DeepLearning** **Model**

In [ ]:
X = df[feature_cols].copy()
y_multi = df["multi_label"].copy()

print("X Shape:", X.shape)
print("Multiclass Labels:", y_multi.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_multi,
    test_size=0.30,
    random_state=42,
    stratify=y_multi
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nClasses in Train:", y_train.nunique())
print("Classes in Validation:", y_val.nunique())
print("Classes in Test:", y_test.nunique())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Number of weights:", len(class_weights))

print(class_weights)

Create PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class CICDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y.values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = CICDataset(
    X_train,
    y_train
)

val_dataset = CICDataset(
    X_val,
    y_val
)

test_dataset = CICDataset(
    X_test,
    y_test
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Transformer Model

In [ ]:
import torch
import torch.nn as nn

class FTTransformer(nn.Module):

    def __init__(self,
                 num_features,
                 num_classes,
                 d_model=128,
                 dropout=0.2):

        super().__init__()

        self.feature_embed = nn.Sequential(
            nn.Linear(num_features, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.block1 = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.block2 = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 34)
        )

    def forward(self, x):

        x = self.feature_embed(x)

        residual = x
        x = self.block1(x)
        x = x + residual

        residual = x
        x = self.block2(x)
        x = x + residual

        return self.classifier(x)

In [ ]:
model = FTTransformer(
    num_features=46,
    num_classes=34
)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [ ]:
import torch

x = torch.randn(10, 5)

print(x)

In [ ]:
X_batch, y_batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(X_batch)

print(outputs.shape)

training the model

In [ ]:
import torch
from sklearn.metrics import accuracy_score

EPOCHS = 10
#Record the start time for training
dl_start_time = time.time()
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # ------------------
    # TRAIN
    # ------------------
    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ------------------
    # VALIDATION
    # ------------------
    model.eval()

    val_loss = 0

    preds = []
    actuals = []

    with torch.inference_mode():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )

            val_loss += loss.item()

            pred = torch.argmax(
                outputs,
                dim=1
            )

            preds.extend(
                pred.cpu().numpy()
            )

            actuals.extend(
                y_batch.cpu().numpy()
            )

    val_loss /= len(val_loader)

    val_acc = accuracy_score(
        actuals,
        preds
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )
# Record the end time for training
dl_end_time = time.time()

# Calculate the elapsed time for training
training_time = dl_end_time - dl_start_time

print("Time taken for training:", training_time, "seconds")

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

start_time = time.time()

model.eval()

preds = []
actuals = []

with torch.inference_mode():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        pred = torch.argmax(outputs, dim=1)

        preds.extend(pred.cpu().numpy())
        actuals.extend(y_batch.numpy())

print(
    classification_report(
        actuals,
        preds,
        digits=4
    )
)
# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
training_time = end_time - start_time

print("Time taken for training:", training_time, "seconds")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

model.eval()

preds_multi = []
actuals_multi = []

with torch.inference_mode():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        pred = torch.argmax(outputs, dim=1)

        preds_multi.extend(
            pred.cpu().numpy()
        )

        actuals_multi.extend(
            y_batch.numpy()
        )

print("Done")

In [ ]:
import pandas as pd

multi_accuracy = accuracy_score(
    actuals_multi,
    preds_multi
)

multi_precision = precision_score(
    actuals_multi,
    preds_multi,
    average='weighted'
)

multi_recall = recall_score(
    actuals_multi,
    preds_multi,
    average='weighted'
)

multi_f1 = f1_score(
    actuals_multi,
    preds_multi,
    average='weighted'
)

multi_macro_f1 = f1_score(
    actuals_multi,
    preds_multi,
    average='macro'
)

multi_df = pd.DataFrame({
    "Model":["Transformer Multiclass"],
    "Classes":[34],
    "Accuracy (%)":[multi_accuracy*100],
    "Precision (%)":[multi_precision*100],
    "Recall (%)":[multi_recall*100],
    "Weighted F1 (%)":[multi_f1*100],
    "Macro F1 (%)":[multi_macro_f1*100]
})

multi_df.round(2)

CNN model

In [ ]:
import torch
import torch.nn as nn

class CNN1D(nn.Module):
    def __init__(self, num_features=46, num_classes=34):
        super().__init__()
        # Input shape: (Batch, 1, 46)
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            # Reduces the feature dimension to 1
            nn.AdaptiveMaxPool1d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Dropout(0.2), # Added dropout to prevent overfitting
            nn.Linear(64, num_classes) # Changed to output 34 classes
        )

    def forward(self, x):
        # x input shape: (batch, 46)
        x = x.unsqueeze(1) # Transform to (batch, 1, 46)
        x = self.conv(x)
        x = x.view(x.size(0), -1) # Flatten to (batch, 32)
        return self.fc(x) # Returns logits

# Example usage:
# model = CNN1D(num_features=46, num_classes=34)

In [ ]:
model = CNN1D(
    num_features=46,
    num_classes=34
)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [ ]:
import torch

x = torch.randn(10, 5)

print(x)

In [ ]:
X_batch, y_batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(X_batch)

print(outputs.shape)

training the CNN model

In [ ]:
import torch
from sklearn.metrics import accuracy_score

EPOCHS = 10
#Record the start time for training
cnn_start_time = time.time()
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # ------------------
    # TRAIN
    # ------------------
    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ------------------
    # VALIDATION
    # ------------------
    model.eval()

    val_loss = 0

    preds = []
    actuals = []

    with torch.inference_mode():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )

            val_loss += loss.item()

            pred = torch.argmax(
                outputs,
                dim=1
            )

            preds.extend(
                pred.cpu().numpy()
            )

            actuals.extend(
                y_batch.cpu().numpy()
            )

    val_loss /= len(val_loader)

    val_acc = accuracy_score(
        actuals,
        preds
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )
# Record the end time for training
cnn_end_time = time.time()

# Calculate the elapsed time for training
cnn_training_time = cnn_end_time - cnn_start_time

print("Time taken for training:", cnn_training_time, "seconds")

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

start_time = time.time()

model.eval()

preds = []
actuals = []

with torch.inference_mode():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        pred = torch.argmax(outputs, dim=1)

        preds.extend(pred.cpu().numpy())
        actuals.extend(y_batch.numpy())

print(
    classification_report(
        actuals,
        preds,
        digits=4
    )
)
# Record the end time for training
end_time = time.time()

# Calculate the elapsed time for training
training_time = end_time - start_time

print("Time taken for training:", training_time, "seconds")

In [ ]:
import pandas as pd

multi_accuracy = accuracy_score(
    actuals,
    preds
)

multi_precision = precision_score(
    actuals,
    preds,
    average='weighted'
)

multi_recall = recall_score(
    actuals,
    preds,
    average='weighted'
)

multi_f1 = f1_score(
    actuals,
    preds,
    average='weighted'
)

multi_macro_f1 = f1_score(
    actuals,
    preds,
    average='macro'
)

multi_df = pd.DataFrame({
    "Model":["Transformer Multiclass"],
    "Classes":[34],
    "Accuracy (%)":[multi_accuracy*100],
    "Precision (%)":[multi_precision*100],
    "Recall (%)":[multi_recall*100],
    "Weighted F1 (%)":[multi_f1*100],
    "Macro F1 (%)":[multi_macro_f1*100]
})

multi_df.round(2)